<a href="https://colab.research.google.com/github/xwang335/Campbell-A/blob/main/data_preprocess_corrected_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preprocessing — Corrected v2

**Based on:** Gu, Kelly & Xiu (2020) "Empirical Asset Pricing via Machine Learning" (SSRN 3159577)

Pipeline:
1. Load 94 stock-level characteristics from `datashare.csv`
2. Query WRDS/CRSP for monthly returns + FF risk-free rate
3. Create `exret_lead1` (next month excess return)
4. Load 8 macro variables (Welch & Goyal 2008), lagged by 1 month
5. Cross-sectionally rank ALL **94** characteristics → map to [-1, 1]
6. Filter to 1957-03 ~ 2016-12, merge macro, save as **Parquet**

**v2 fixes vs. v1:**
- `sic2` excluded from rank normalization (`CHAR_COLS_94` now correctly = 94 not 95)
- Save directly as Parquet (replaces slow 6 GB CSV conversion step)

## 0. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q wrds gspread google-api-python-client pyarrow

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.6 MB/s eta 0:00:00


In [2]:
import wrds
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print('All imports OK')

All imports OK


## 1. Load datashare.csv (94 Stock-Level Characteristics)

Paper Section 3.1: "a large collection of stock-level predictive characteristics based on the cross section
of stock returns literature. These include 94 characteristics (61 updated annually, 13 quarterly, 20 monthly).
In addition, we include 74 industry dummies corresponding to the first two digits of SIC codes."

> Note: `datashare.csv` contains 94 characteristics + `sic2` (industry code). `sic2` is **not** a continuous
> characteristic and should **not** be rank-normalized. It is used to construct 74 industry dummies.

In [3]:
csv_path = "/content/drive/MyDrive/datashare.csv"
df = pd.read_csv(csv_path)
df['DATE'] = pd.to_datetime(df['DATE'], format='%Y%m%d')

print(f"datashare shape: {df.shape}")
print(f"Date range:      {df['DATE'].min().date()} to {df['DATE'].max().date()}")
print(f"Columns (first 5): {list(df.columns[:5])}")
print(f"Total cols - 2 (permno/DATE) = {df.shape[1]-2} (should be 95 = 94 chars + sic2)")

datashare shape: (4117300, 97)
Date range:      1957-01-31 to 2021-12-31
Columns (first 5): ['permno', 'DATE', 'mvel1', 'beta', 'betasq']
Total cols - 2 (permno/DATE) = 95 (should be 95 = 94 chars + sic2)


## 2. Query WRDS/CRSP

In [4]:
db = wrds.Connection()

Enter your WRDS username [root]:zixian_zhou
Enter your password:··········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: n
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [5]:
start_date = df['DATE'].min()
end_date   = df['DATE'].max()

query = f"""
SELECT
    b.permno,
    b.date,
    b.ret,
    c.rf,
    (b.ret - c.rf) AS exret
FROM crsp.msf b
LEFT JOIN ff.factors_monthly c ON
    EXTRACT(YEAR  FROM b.date) = EXTRACT(YEAR  FROM c.date)
    AND EXTRACT(MONTH FROM b.date) = EXTRACT(MONTH FROM c.date)
WHERE b.date >= '{start_date}' AND b.date <= '{end_date}'
  AND b.ret IS NOT NULL
"""

df_msf = db.raw_sql(query)
df_msf['date'] = pd.to_datetime(df_msf['date'])
print(f"CRSP rows: {len(df_msf):,}")

CRSP rows: 4,318,501


In [6]:
# Merge datashare with CRSP on (permno, DATE)
df_exret = pd.merge(
    df, df_msf,
    left_on=['permno', 'DATE'],
    right_on=['permno', 'date'],
    how='inner'
).drop(columns=['date'])   # drop duplicate date col

print(f"After merge: {len(df_exret):,} rows")

After merge: 4,096,791 rows


## 3. Basic Filters

In [7]:
n0 = len(df_exret)
df_exret = df_exret[df_exret['ret'].notna()].copy()
df_exret = df_exret[df_exret['mvel1'].notna()].copy()
df_exret.reset_index(drop=True, inplace=True)

print(f"Dropped (ret/mvel1 NaN): {n0 - len(df_exret):,}  |  Remaining: {len(df_exret):,}")

Dropped (ret/mvel1 NaN): 2,982  |  Remaining: 4,093,809


## 4. Create Target Variable `exret_lead1`

Paper Equation (1): $r_{i,t+1} = \mathrm{E}_t(r_{i,t+1}) + \epsilon_{i,t+1}$

The prediction target is the **next month's excess return**.
Created by shifting `exret` by -1 within each `permno`.
Must be done **before** date filtering to capture valid cross-permno boundaries.

In [8]:
df_exret = df_exret.sort_values(['permno', 'DATE']).reset_index(drop=True)
df_exret['exret_lead1'] = df_exret.groupby('permno')['exret'].shift(-1)

print(f"exret_lead1 NaN: {df_exret['exret_lead1'].isna().sum():,}  (last obs of each permno)")

exret_lead1 NaN: 32,750  (last obs of each permno)


## 5. Filter to Paper Sample Window: 1957-03 ~ 2016-12

Paper Section 3.1: "Our sample begins in March 1957 (the start date of the S&P 500) and ends in December 2016."

In [9]:
n0 = len(df_exret)

df_exret = df_exret[
    (df_exret['DATE'] >= pd.Timestamp('1957-03-31')) &
    (df_exret['DATE'] <= pd.Timestamp('2016-12-31'))
].copy()
df_exret = df_exret.dropna(subset=['exret_lead1']).reset_index(drop=True)

print(f"Dropped (date/target filter): {n0 - len(df_exret):,}")
print(f"Final rows: {len(df_exret):,}")
print(f"Date range: {df_exret['DATE'].min().date()} to {df_exret['DATE'].max().date()}")

Dropped (date/target filter): 381,001
Final rows: 3,712,808
Date range: 1957-04-30 to 2016-12-30


## 6. Cross-Sectional Rank Normalization (94 Characteristics Only)

Paper footnote 29: *"We cross-sectionally rank **all stock characteristics** period-by-period
and map these ranks into the [-1,1] interval."*

Paper footnote 30: *"Missing characteristics are replaced with the cross-sectional median."*

**Important:** `sic2` is excluded — it is a categorical industry code used to construct 74
industry dummy variables, not a continuous characteristic to be rank-normalized.

In [10]:
# Columns that are NOT stock characteristics
# sic2 is explicitly excluded — it is categorical (industry code), not a continuous characteristic
NON_CHAR_COLS = {'permno', 'DATE', 'ret', 'rf', 'exret', 'exret_lead1', 'sic2'}
CHAR_COLS_94  = [c for c in df_exret.columns if c not in NON_CHAR_COLS]

print(f"Characteristic columns to rank-normalize: {len(CHAR_COLS_94)}  (should be 94)")
print(f"First 5:  {CHAR_COLS_94[:5]}")
print(f"Last 5:   {CHAR_COLS_94[-5:]}")
print(f"sic2 in CHAR_COLS_94: {'sic2' in CHAR_COLS_94}  (should be False)")

Characteristic columns to rank-normalize: 94  (should be 94)
First 5:  ['mvel1', 'beta', 'betasq', 'chmom', 'dolvol']
Last 5:   ['maxret', 'retvol', 'std_dolvol', 'std_turn', 'zerotrade']
sic2 in CHAR_COLS_94: False  (should be False)


In [11]:
def rank_norm_median(s: pd.Series) -> pd.Series:
    """
    Cross-sectional rank normalization (per month):
    1) Fill NaN with cross-sectional median (paper fn.30)
    2) Rank with average ties
    3) Map to [-1, 1]:  mapped = (rank / (N+1)) * 2 - 1
    """
    filled = s.fillna(s.median(skipna=True)).fillna(0.0)
    ranks  = filled.rank(method='average')
    n      = ranks.count()
    return (ranks / (n + 1)) * 2 - 1 if n > 0 else filled


print("Rank-normalizing 94 characteristics cross-sectionally by DATE...")
df_exret[CHAR_COLS_94] = (
    df_exret.groupby('DATE')[CHAR_COLS_94]
    .transform(rank_norm_median)
)
print("Done!")

# Sanity check
for c in ['mvel1', 'bm', 'mom12m']:
    print(f"  {c:8s}  min={df_exret[c].min():.4f}  max={df_exret[c].max():.4f}  mean={df_exret[c].mean():+.2e}")

Rank-normalizing 94 characteristics cross-sectionally by DATE...
Done!
  mvel1     min=-0.9998  max=0.9998  mean=+2.15e-17
  bm        min=-0.9990  max=0.9990  mean=+2.66e-18
  mom12m    min=-0.9955  max=0.9956  mean=+2.27e-18


## 7. Load 8 Macroeconomic Predictors (Welch & Goyal 2008)

Variables: `tbl`, `d/p`, `e/p`, `b/m`, `tms`, `dfy`, `ntis`, `svar`

Lagged by 1 month to avoid look-ahead bias.

In [12]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
from googleapiclient.discovery import build

creds, _ = default()
gc       = gspread.authorize(creds)
sh       = gc.open("Data2024")

drive_svc   = build('drive', 'v3', credentials=creds)
export_path = "/content/Data2024_export.xlsx"

req = drive_svc.files().export_media(
    fileId=sh.id,
    mimeType='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
)
with open(export_path, "wb") as f:
    f.write(req.execute())

df_macro = pd.read_excel(export_path, sheet_name="Monthly")
print(f"Macro sheet shape: {df_macro.shape}")

Macro sheet shape: (1848, 57)


In [13]:
MACRO_COLS = ['tbl', 'd/p', 'e/p', 'b/m', 'tms', 'dfy', 'ntis', 'svar']

df_macro = df_macro.sort_values('yyyymm').reset_index(drop=True)
df_macro[MACRO_COLS] = df_macro[MACRO_COLS].shift(1)   # lag by 1 month

df_macro['year_month'] = (
    pd.to_datetime(df_macro['yyyymm'], format='%Y%m').dt.to_period('M')
)
df_macro = df_macro[['yyyymm'] + MACRO_COLS + ['year_month']]

print(f"Macro rows after lag: {len(df_macro)}")

Macro rows after lag: 1848


## 8. Merge Stock Data with Macro

In [14]:
df_exret['year_month'] = df_exret['DATE'].dt.to_period('M')

df_merged = pd.merge(df_exret, df_macro, on='year_month', how='left')
df_merged = df_merged.drop(columns=['yyyymm', 'year_month'])   # ← must assign back

print(f"Merged shape: {df_merged.shape}")
print(f"Macro NaN (within 1957-2016, expect 0):")
print(df_merged[MACRO_COLS].isna().sum().to_string())

Merged shape: (3712808, 109)
Macro NaN (within 1957-2016, expect 0):
tbl     0
d/p     0
e/p     0
b/m     0
tms     0
dfy     0
ntis    0
svar    0


## 9. Forward-fill sic2 (Industry Code)

sic2 is kept as-is (not rank-normalized). Forward-fill within permno to handle gaps.

In [15]:
if 'sic2' in df_merged.columns:
    n_nan = df_merged['sic2'].isna().sum()
    df_merged['sic2'] = df_merged.groupby('permno')['sic2'].ffill().bfill()
    print(f"sic2 NaN: {n_nan:,} → {df_merged['sic2'].isna().sum():,}")
else:
    print("Warning: sic2 not found")

sic2 NaN: 245,649 → 0


## 10. Final Summary

In [16]:
print("=" * 65)
print("FINAL PREPROCESSED DATA")
print("=" * 65)
print(f"Shape:          {df_merged.shape}")
print(f"Date range:     {df_merged['DATE'].min().date()} → {df_merged['DATE'].max().date()}")
print(f"Unique permnos: {df_merged['permno'].nunique():,}")
print(f"Unique months:  {df_merged['DATE'].nunique()}")
print()
print(f"exret_lead1  NaN: {df_merged['exret_lead1'].isna().sum()}")
print(f"             mean={df_merged['exret_lead1'].mean():.6f}  std={df_merged['exret_lead1'].std():.6f}")
print()
print("Rank-normalized check (values must be in [-1, 1]):")
for c in ['mvel1', 'bm', 'mom12m']:
    print(f"  {c:10s}  [{df_merged[c].min():.4f}, {df_merged[c].max():.4f}]")
print()
print("sic2 sample (raw, NOT rank-normalized):")
print(df_merged['sic2'].value_counts().head(5).to_string())

FINAL PREPROCESSED DATA
Shape:          (3712808, 109)
Date range:     1957-04-30 → 2016-12-30
Unique permnos: 29,825
Unique months:  717

exret_lead1  NaN: 0
             mean=0.007313  std=0.172393

Rank-normalized check (values must be in [-1, 1]):
  mvel1       [-0.9998, 0.9998]
  bm          [-0.9990, 0.9990]
  mom12m      [-0.9955, 0.9956]

sic2 sample (raw, NOT rank-normalized):
sic2
67.0    353505
60.0    302308
73.0    264641
28.0    230205
36.0    225301


## 11. Save to Google Drive as Parquet

Parquet is ~1.9 GB vs ~6.3 GB for CSV — much faster to load in the OLS3 notebook.

In [17]:
db.close()
print("WRDS connection closed.")

save_dir = '/content/drive/MyDrive/industry_project'
os.makedirs(save_dir, exist_ok=True)

parquet_path = os.path.join(save_dir, 'preprocess_data.parquet')
df_merged.to_parquet(parquet_path, index=False, engine='pyarrow')

print(f"Saved Parquet: {parquet_path}")
print(f"File size:     {os.path.getsize(parquet_path) / 1e9:.2f} GB")

WRDS connection closed.
Saved Parquet: /content/drive/MyDrive/industry_project/preprocess_data.parquet
File size:     1.93 GB


---
## Summary of All Fixes (Original → v1 → v2)

| Issue | Original | v1 | v2 (This) |
|---|---|---|---|
| **sic2 rank-normalized** | ✅ normalized | ✅ normalized (bug) | ✅ **Excluded** — sic2 is categorical |
| **CHAR_COLS count** | 94 (all 94+sic2) | 95 (bug) | **94 (correct)** |
| **exret_lead1** | Not created | ✅ Created | ✅ Created |
| **Date filter end** | None (to 2021) | ✅ 2016-12 | ✅ 2016-12 |
| **Macro loading** | pd.read_excel (bug) | ✅ gspread | ✅ gspread |
| **Macro dropna** | All 57 cols (bug) | ✅ Fixed | ✅ Fixed |
| **drop() assigned back** | Bug | ✅ Fixed | ✅ Fixed |
| **Save format** | CSV 6.3 GB + convert | CSV then parquet | **Parquet directly** |